In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression

from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv("cleaned_data.csv")
display(df.head())
print(df.shape)
print(df.dtypes)

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,51000,E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,10300
1,Hyundai,Palisade SEL,2021,34742,Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,38005
2,Lexus,RX 350 RX 350,2022,22372,Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,54598
3,INFINITI,Q50 Hybrid Sport,2015,88900,Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,15500
4,Audi,Q3 45 S line Premium Plus,2021,9835,Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,34999


(4009, 12)
brand           object
model           object
model_year       int64
milage           int64
fuel_type       object
engine          object
transmission    object
ext_col         object
int_col         object
accident        object
clean_title     object
price            int64
dtype: object


In [ ]:
y_reg = df["price"]
y_clf = (y_reg > y_reg.median()).astype(int)
X = df.drop(columns=["price"])
print("Regression Target:", y_reg.name)
print("\nClassification Target Distribution:")
print(y_clf.value_counts())

Regression Target: price

Classification Target Distribution:
price
0    2011
1    1998
Name: count, dtype: int64


In [ ]:
X = pd.get_dummies(X, drop_first=True)
print("Encoded Feature Matrix Shape:", X.shape)
display(X.head())

Encoded Feature Matrix Shape: (4009, 3641)


,model_year,milage,brand_Alfa,brand_Aston,brand_Audi,brand_BMW,brand_Bentley,brand_Bugatti,brand_Buick,brand_Cadillac,...,int_col_Tupelo,int_col_Very Light Cashmere,int_col_WHITE,int_col_Walnut,int_col_Whisper Beige,int_col_White,int_col_White / Brown,int_col_Yellow,int_col_–,accident_None reported
0,2013,51000,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,2021,34742,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,2022,22372,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
3,2015,88900,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
4,2021,9835,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True


In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(
    random_state=42,
    max_iter=1000
)

log_reg.fit(X_train_scaled, y_clf_train)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


In [ ]:
X_train, X_test, y_reg_train, y_reg_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)
_, _, y_clf_train, y_clf_test = train_test_split(
    X, y_clf, test_size=0.2, random_state=42
)
print("Training Set:", X_train.shape)
print("Testing Set :", X_test.shape)

Training Set: (3207, 3641)
Testing Set : (802, 3641)


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Feature scaling completed successfully.")

Feature scaling completed successfully.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    roc_curve
)

from sklearn.model_selection import (
    cross_val_score,
    StratifiedKFold,
    GridSearchCV
)

from sklearn.pipeline import make_pipeline

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

import joblib

In [ ]:
dt_default = DecisionTreeClassifier(random_state=42)

dt_default.fit(X_train_scaled, y_clf_train)

train_pred = dt_default.predict(X_train_scaled)
test_pred = dt_default.predict(X_test_scaled)

train_acc = accuracy_score(y_clf_train, train_pred)
test_acc = accuracy_score(y_clf_test, test_pred)

print("Decision Tree (Default)")
print("Training Accuracy:", train_acc)
print("Testing Accuracy :", test_acc)

Decision Tree (Default)
Training Accuracy: 1.0
Testing Accuracy : 0.8541147132169576


In [ ]:
dt_controlled = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

dt_controlled.fit(X_train_scaled, y_clf_train)

train_pred = dt_controlled.predict(X_train_scaled)
test_pred = dt_controlled.predict(X_test_scaled)

print("Training Accuracy:",
      accuracy_score(y_clf_train, train_pred))

print("Testing Accuracy:",
      accuracy_score(y_clf_test, test_pred))

Training Accuracy: 0.8437792329279701
Testing Accuracy: 0.8416458852867831


In [ ]:
gini_tree = DecisionTreeClassifier(
    criterion='gini',
    max_depth=5,
    random_state=42
)

entropy_tree = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=5,
    random_state=42
)

gini_tree.fit(X_train_scaled, y_clf_train)
entropy_tree.fit(X_train_scaled, y_clf_train)

gini_acc = accuracy_score(
    y_clf_test,
    gini_tree.predict(X_test_scaled)
)

entropy_acc = accuracy_score(
    y_clf_test,
    entropy_tree.predict(X_test_scaled)
)

print("Gini Accuracy:", gini_acc)
print("Entropy Accuracy:", entropy_acc)

Gini Accuracy: 0.8366583541147132
Entropy Accuracy: 0.8229426433915212


In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf.fit(X_train_scaled, y_clf_train)

train_pred = rf.predict(X_train_scaled)
test_pred = rf.predict(X_test_scaled)

train_acc = accuracy_score(y_clf_train, train_pred)
test_acc = accuracy_score(y_clf_test, test_pred)

rf_probs = rf.predict_proba(X_test_scaled)[:,1]

auc = roc_auc_score(y_clf_test, rf_probs)

print("Training Accuracy:", train_acc)
print("Testing Accuracy:", test_acc)
print("ROC AUC:", auc)

Training Accuracy: 0.8671655753040225
Testing Accuracy: 0.8541147132169576
ROC AUC: 0.9245040034341812


In [ ]:
importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

print(importance.head())

                     Feature  Importance
1                     milage    0.187528
0                 model_year    0.135741
3640  accident_None reported    0.038801
3143        transmission_A/T    0.026881
3145  transmission_Automatic    0.026152


In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb.fit(X_train_scaled, y_clf_train)

train_acc = accuracy_score(
    y_clf_train,
    gb.predict(X_train_scaled)
)

test_acc = accuracy_score(
    y_clf_test,
    gb.predict(X_test_scaled)
)

gb_auc = roc_auc_score(
    y_clf_test,
    gb.predict_proba(X_test_scaled)[:,1]
)

print(train_acc)
print(test_acc)
print(gb_auc)

0.8846273776114749
0.8628428927680798
0.9385175784044745


In [ ]:
lowest_features = importance.tail(5)["Feature"]

print(lowest_features)

705        model_Explorer Sport Trac XLT
708                 model_Explorer sport
1842                  model_X2 xDrive28i
1843                      model_X3 M AWD
1836    model_Wrangler Unlimited Sport S
Name: Feature, dtype: object


In [ ]:
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

In [ ]:
X_train_reduced = X_train_scaled.drop(columns=lowest_features)
X_test_reduced = X_test_scaled.drop(columns=lowest_features)

rf_reduced = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_reduced.fit(X_train_reduced, y_clf_train)

auc_full = roc_auc_score(
    y_clf_test,
    rf.predict_proba(X_test_scaled)[:,1]
)

auc_reduced = roc_auc_score(
    y_clf_test,
    rf_reduced.predict_proba(X_test_reduced)[:,1]
)

print("Full Model AUC:", auc_full)
print("Reduced Model AUC:", auc_reduced)

Full Model AUC: 0.9245040034341812
Reduced Model AUC: 0.9208645178148156


In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

models = {
    "Logistic Regression": log_reg,
    "Decision Tree": dt_controlled,
    "Random Forest": rf,
    "Gradient Boosting": gb
}

results = []

for name, model in models.items():

    scores = cross_val_score(
        model,
        X_train_scaled,
        y_clf_train,
        cv=cv,
        scoring='roc_auc'
    )

    results.append([
        name,
        scores.mean(),
        scores.std()
    ])

cv_results = pd.DataFrame(
    results,
    columns=["Model","Mean AUC","Std"]
)

print(cv_results)

                 Model  Mean AUC       Std
0  Logistic Regression  0.946259  0.005975
1        Decision Tree  0.883538  0.006022
2        Random Forest  0.923266  0.006880
3    Gradient Boosting  0.931893  0.003550


In [48]:
pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

param_grid = {
    'randomforestclassifier__n_estimators': [50],
    'randomforestclassifier__max_depth': [5, 10],
    'randomforestclassifier__min_samples_leaf': [1]
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=1,
    verbose=2
)

grid.fit(X_train, y_clf_train)

print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

Fitting 5 folds for each of 2 candidates, totalling 10 fits
[CV] END randomforestclassifier__max_depth=5, randomforestclassifier__min_samples_leaf=1, randomforestclassifier__n_estimators=50; total time=   3.7s
[CV] END randomforestclassifier__max_depth=5, randomforestclassifier__min_samples_leaf=1, randomforestclassifier__n_estimators=50; total time=   2.3s
[CV] END randomforestclassifier__max_depth=5, randomforestclassifier__min_samples_leaf=1, randomforestclassifier__n_estimators=50; total time=   1.7s
[CV] END randomforestclassifier__max_depth=5, randomforestclassifier__min_samples_leaf=1, randomforestclassifier__n_estimators=50; total time=   1.7s
[CV] END randomforestclassifier__max_depth=5, randomforestclassifier__min_samples_leaf=1, randomforestclassifier__n_estimators=50; total time=   1.7s
[CV] END randomforestclassifier__max_depth=10, randomforestclassifier__min_samples_leaf=1, randomforestclassifier__n_estimators=50; total time=   2.2s
[CV] END randomforestclassifier__max_de

In [53]:
best_pipeline = grid.best_estimator_

In [54]:
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]

results = []

for frac in fractions:

    n = int(frac * len(X_train))

    X_sub = X_train.iloc[:n]
    y_sub = y_clf_train.iloc[:n]

    best_pipeline.fit(X_sub, y_sub)

    train_auc = roc_auc_score(
        y_sub,
        best_pipeline.predict_proba(X_sub)[:, 1]
    )

    test_auc = roc_auc_score(
        y_clf_test,
        best_pipeline.predict_proba(X_test)[:, 1]
    )

    results.append([frac, train_auc, test_auc])

learning_curve = pd.DataFrame(
    results,
    columns=["Training Fraction", "Training AUC", "Test AUC"]
)

print(learning_curve)

   Training Fraction  Training AUC  Test AUC
0                0.2      0.966299  0.890277
1                0.4      0.954298  0.911847
2                0.6      0.944954  0.913847
3                0.8      0.945421  0.919944
4                1.0      0.944016  0.923869


In [55]:
import joblib

joblib.dump(best_pipeline, "best_model.pkl")

print("Model saved successfully.")

Model saved successfully.


In [56]:
loaded_model = joblib.load("best_model.pkl")

sample = X_test.iloc[:2]

predictions = loaded_model.predict(sample)

print(predictions)

[0 0]
